# មេរៀន ០៥ - Agentic RAG


## ការតំឡើង

កំណត់ត្រានេះបង្ហាញពីលំនាំ Agentic RAG (Retrieval-Augmented Generation) ដោយប្រើ Microsoft Agent Framework។

**សម្គាល់ត្រូវមានៈ**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ចំណុចចប់សេវាពី Azure AI Search របស់អ្នក
- `AZURE_SEARCH_API_KEY` — កូនសោ API របស់ Azure AI Search របស់អ្នក
- ការតំឡើង Azure OpenAI តាមរយៈអថេរបរិស្ថាន
- Azure CLI បានផ្ទៀងផ្ទាត់ស身份 (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Agentic RAG ជាអ្វី?

RAG ធម្មតាដំណើរការតាមបណ្តាញថ្នាក់ដែរមានផ្លូវថាសថេរ៖ ទាញយកឯកសារ បន្ទាប់មកបង្កើតការឆ្លើយតប។ **Agentic RAG** ចូលចិត្តមុខសញ្ញាមួយកំឡុងពេលឲ្យភ្នាក់ងារជ្រើសរើសដោយស្វ័យប្រវត្តិ **ពេលណា** និង **របៀបណា** ក្នុងការទាញយកព័ត៌មាន។

ជាមួយ Agentic RAG ភ្នាក់ងារអាច៖
- **សម្រេចចិត្ត** ថាតើត្រូវការលុំព័ត៌មានមុនពេលឆ្លើយសំណួរ ឬអត់
- **ជ្រើសរើស** ប្រភពទិន្នន័យ ឬឧបករណ៍ដែលត្រូវពិនិត្យ
- **វាយតម្លៃ** លទ្ធផលទាញយកបាន និងអាចបន្តទាញយកបន្ថែមបើការប្រលងដំបូងមិនគ្រប់គ្រាន់
- **បញ្ចូល** ព័ត៌មានពីជំហានទាញយកជាច្រើនទៅជា​ ការឆ្លើយតបដែលសមរម្យ

នេះធ្វើឲ្យភ្នាក់ងារមានភាពបត់បែននិងត្រឹមត្រូវច្រើនជាងបណ្តាញថ្នាក់ទាញយកបន្ទាប់បង្កើតថេរ។


## ការបង្កើតឧបករណ៍ស្វែងរក

នៅក្នុង Agentic RAG ដាតាបណ្ដាញក្រៅត្រូវបានប័ណ្ណាដូចជា **ឧបករណ៍** ដែលភ្នាក់ងារអាចហៅបានពេលត្រូវការ។ វាធ្វើឱ្យភ្នាក់ងារប្រើការស្វែងរកជាអំពើមួយទៀតដែលវាអាចអនុវត្តបាន មិនមែនជាជំហានបំណាច់ទេ។

ខាងក្រោមនេះ យើងបានកំណត់មូលដ្ឋានចំណេះដឹងដំណើរកម្សាន្ត និងបង្ហាញវាជាឧបករណ៍ដែលភ្នាក់ងារអាចហៅដើម្បីស្វែងរកព័ត៌មានគោលដៅ។


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## ការស្ថាបនាតំណាង RAG

ឥឡូវនេះ យើងបង្កើតតំណាងម្នាក់ដែលត្រូវបានណែនាំឱ្យ **រក្សាទុកព័ត៌មានជាមុនសិនពេលឆ្លើយតប**។ តំណាងនេះប្រើឧបករណ៍ `search_travel_knowledge` ដើម្បីមានមូលដ្ឋាននៃការឆ្លើយតបដោយផ្អែកលើមូលដ្ឋានចំណេះដឹង ជំនួសពីការជឿទុកចិត្តលើទិន្នន័យបណ្តុះបណ្តាលផ្ទាល់ខ្លួន។


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ការស្វែងរកជាដំណើរការ​ច្រើនដង — លំនាំអ្នកបង្កើត-អ្នកពិនិត្យ

អត្ថប្រយោជន៍សំខាន់មួយនៃ Agentic RAG គឺ **ការស្វែងរកជាដំណើរការ​ច្រើនដង**។ អ្នកតំណាងអាចបំពេញការស្វែងរកជាច្រើនដង ដើម្បីផ្ទៀងផ្ទាត់ កែសម្រួល ឬពង្រីកលទ្ធផលដើមរបស់ខ្លួន — ដូចជាការដំណើរការ "អ្នកបង្កើត-អ្នកពិនិត្យ"៖

1. **ជំហានអ្នកបង្កើត**: អ្នកតំណាងទាញយកពត៌មានដើម ហើយរៀបចំចម្លើយមួយ។
2. **ជំហានអ្នកពិនិត្យ**: អ្នកតំណាងបំពេញការស្វែងរកបន្ថែម ដើម្បីផ្ទៀងផ្ទាត់ព័ត៌មាន ឬបន្ថែមតំបន់ទាល់។

ខាងក្រោមនេះ អ្នកតំណាងត្រូវបានសួរបញ្ហាមួយ ដែលត្រូវការប្រៀបធៀបកន្លែងគោលដៅជាច្រើន បណ្តាលឲ្យវាស្វែងរកច្រើនដង។


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## សង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀបបង្កើតប្រព័ន្ធ **Agentic RAG** ដោយប្រើ Microsoft Agent Framework៖

- **Agentic RAG** អនុញ្ញាតឲ្យភ្នាក់ងារជូនដំណឹងដោយស្វ័យប្រវត្តិនៅពេលស្វែងយល់ព័ត៌មាន ដែលធ្វើឲ្យការស្វែងយល់មានលក្ខណៈបត់បែន មិនមែនថេរឡើយ។
- **ឧបករណ៍ជាតិប្រភពទិន្នន័យ**៖ មូលដ្ឋានចំណេះដឹងខាងក្រៅ (ដូចជា Azure AI Search) ត្រូវបានបើកដូចជាឧបករណ៍ដែលភ្នាក់ងារអាចប្រើបាន។
- **ការស្វែងយល់តាមដំណាក់កាល**៖ លំនាំ maker-checker អនុញ្ញាតឲ្យភ្នាក់ងារធ្វើការស្វែងយល់ច្រើនដង — ស្វែងរក, វាយតម្លៃ, និងកែលម្អ — មុនពេលបញ្ចេញចម្លើយចុងក្រោយ។

ក្នុងការផលិត អ្នកនឹងផ្លាស់ប្តូរមូលដ្ឋាន `TRAVEL_KNOWLEDGE_BASE` ដែលមាននៅក្នុងចំណាំទៅជាការស៊ើបអង្កេតនៃ Azure AI Search ពិតប្រាកដ ដើម្បីដោះសោនការស្វែងយល់ឯកសារធំៗខាងដំណើរទេសចរណ៍។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
